# ETS Generator

Generate Exponential Smoothing (ETS) time series with configurable error, trend, and seasonal components. Supports additive and multiplicative formulations, damped trends, state export, and Box-Cox transformations.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import ETSGenerator

## Example 1: Damped Additive Holt-Winters ETS(A,Ad,A)

An additive model with weekly seasonality and a damped trend.

In [ ]:
params = {
    "min_length": 200,
    "max_length": 200,
    "freq": "D",
    "error_type": "add",
    "trend_type": "add",
    "seasonal_type": "add",
    "seasonal_period": 7,
    "level": 100.0,
    "trend": 0.5,
    "alpha": 0.3,
    "beta": 0.1,
    "gamma": 0.1,
    "damped": True,
    "phi": 0.95,
    "noise_std": 2.0,
    "seed": 123,
}

generator = ETSGenerator(engine="polars", **params)

### Model Information

In [ ]:
model_info = generator.get_model_info()
print(f"Model: {model_info['model']}")
print(f"Initial level: {model_info['level']}")
print(f"Initial trend: {model_info['trend']}")
print(f"Smoothing parameters: alpha={model_info['alpha']}, beta={model_info['beta']}, gamma={model_info['gamma']}")
print(f"Damping parameter: phi={model_info['phi']}")

### Generate and Inspect Data

In [ ]:
df = generator.generate(n_series=3)
print(f"Generated {df['unique_id'].n_unique()} time series")
print(f"Total observations: {len(df)}")
df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("ETS(A,Ad,A) Damped Additive Holt-Winters")
ax.legend()
plt.tight_layout()
plt.show()

### Statistics by Series

In [ ]:
df.group_by("unique_id").agg(
    [
        pl.col("y").count().alias("count"),
        pl.col("y").min().alias("min_value"),
        pl.col("y").max().alias("max_value"),
        pl.col("y").mean().alias("mean_value"),
        pl.col("y").std().alias("std_value"),
    ]
).sort("unique_id")

## Example 2: ETS(A,A,A) with State Export

Generate data and extract the underlying level, trend, and seasonal state components.

In [ ]:
params_states = {
    "min_length": 50,
    "max_length": 50,
    "freq": "D",
    "error_type": "add",
    "trend_type": "add",
    "seasonal_type": "add",
    "seasonal_period": 7,
    "level": 100.0,
    "trend": 1.0,
    "alpha": 0.3,
    "beta": 0.1,
    "gamma": 0.1,
    "noise_std": 1.0,
    "seed": 42,
}

generator_states = ETSGenerator(engine="polars", **params_states)
obs_df, states_df = generator_states.generate_with_states(n_series=1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in obs_df["unique_id"].unique().to_list():
    series = obs_df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("ETS(A,A,A) with State Export")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Observations DataFrame:")
obs_df.head(10)

In [ ]:
print("States DataFrame (level, trend, seasonal components):")
states_df.head(10)

## Example 3: Multiplicative Holt-Winters ETS(M,A,M)

A multiplicative error and seasonal model with monthly frequency. All values remain positive.

In [ ]:
params_mul = {
    "min_length": 200,
    "max_length": 200,
    "freq": "MS",
    "error_type": "mul",
    "trend_type": "add",
    "seasonal_type": "mul",
    "seasonal_period": 12,
    "level": 100.0,
    "trend": 1.0,
    "alpha": 0.3,
    "beta": 0.1,
    "gamma": 0.1,
    "noise_std": 0.1,
    "seed": 456,
}

generator_mul = ETSGenerator(engine="polars", **params_mul)
df_mul = generator_mul.generate(n_series=1)

print(f"Model: {generator_mul.get_model_info()['model']}")
print(f"All values positive: {(df_mul['y'] > 0).all()}")
df_mul.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_mul["unique_id"].unique().to_list():
    series = df_mul.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("ETS(M,A,M) Multiplicative Holt-Winters")
ax.legend()
plt.tight_layout()
plt.show()

## Example 4: Simple Exponential Smoothing ETS(A,N,N)

A model with no trend and no seasonality -- just level smoothing with additive errors.

In [ ]:
params_ses = {
    "min_length": 100,
    "max_length": 100,
    "freq": "D",
    "error_type": "add",
    "trend_type": None,
    "seasonal_type": None,
    "level": 50.0,
    "alpha": 0.2,
    "noise_std": 5.0,
    "seed": 789,
}

generator_ses = ETSGenerator(engine="polars", **params_ses)
df_ses = generator_ses.generate(n_series=1)

print(f"Model: {generator_ses.get_model_info()['model']}")
print(f"Mean: {df_ses['y'].mean():.2f}")
print(f"Std: {df_ses['y'].std():.2f}")
print(f"Min: {df_ses['y'].min():.2f}")
print(f"Max: {df_ses['y'].max():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_ses["unique_id"].unique().to_list():
    series = df_ses.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("ETS(A,N,N) Simple Exponential Smoothing")
ax.legend()
plt.tight_layout()
plt.show()

## Example 5: ETS with Box-Cox Transformation

Apply a Box-Cox transformation (lambda=0.5, square root) to the generated series.

In [ ]:
params_boxcox = {
    "min_length": 100,
    "max_length": 100,
    "freq": "D",
    "error_type": "add",
    "trend_type": "add",
    "seasonal_type": None,
    "level": 100.0,
    "trend": 0.5,
    "alpha": 0.3,
    "beta": 0.1,
    "noise_std": 0.5,
    "box_cox_lambda": 0.5,
    "seed": 111,
}

generator_boxcox = ETSGenerator(engine="polars", **params_boxcox)
df_boxcox = generator_boxcox.generate(n_series=1)

print(f"Model: {generator_boxcox.get_model_info()['model']}")
print(f"Box-Cox lambda: {generator_boxcox.get_model_info()['box_cox_lambda']}")
print(f"Mean: {df_boxcox['y'].mean():.2f}")
print(f"Std: {df_boxcox['y'].std():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_boxcox["unique_id"].unique().to_list():
    series = df_boxcox.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("ETS with Box-Cox Transformation")
ax.legend()
plt.tight_layout()
plt.show()